In [1]:
pip install --upgrade gensim

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
import nltk
import nltk.corpus
import nltk.tokenize
import string
from string import punctuation
import torch
import gensim
from gensim.models.word2vec import Word2Vec
import torch.nn as nn
import torch.nn.functional as F
import scipy
import pickle
import torch.utils.data as data_utils
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

In [4]:
# Устанавливаем gensim (если еще не установлен)
!pip install --upgrade gensim

import gensim
import gensim.downloader as download_api
import numpy as np
import pandas as pd
from tqdm import tqdm

print("Доступные модели для скачивания:")
print(download_api.info()['models'].keys())
print("\n" + "="*80)

# Скачиваем русскую модель word2vec (обученную на Национальном корпусе русского языка)
print("Скачиваем модель 'word2vec-ruscorpora-300'...")
print("Это может занять несколько минут (модель весит ~1.5 ГБ)")
print("Пожалуйста, подожди...")

try:
    # Скачиваем модель (загружается в кэш Gensim)
    model = download_api.load('word2vec-ruscorpora-300')
    
    print("✅ Модель успешно загружена!")
    print(f"Размерность векторов: {model.vector_size}")
    print(f"Количество слов в словаре: {len(model.key_to_index)}")
    
    # Покажем примеры слов в модели
    words = list(model.key_to_index.keys())[:10]
    print(f"Примеры слов: {words}")
    
    # Обрати внимание - слова в формате "слово_ЧАСТЬ_РЕЧИ"
    print("\nФормат слов: слово_ЧАСТЬ_РЕЧИ (например, 'работа_NOUN')")
    
except Exception as e:
    print(f"❌ Ошибка при загрузке модели: {e}")
    print("\nАльтернативный вариант - скачать модель вручную:")
    print("1. Перейди на https://rusvectores.org/ru/models/")
    print("2. Скачай модель 'web_0_300_20.bin'")
    print("3. Положи файл в папку с ноутбуком")
    print("4. Запусти код снова")

Доступные модели для скачивания:


dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

Скачиваем модель 'word2vec-ruscorpora-300'...
Это может занять несколько минут (модель весит ~1.5 ГБ)
Пожалуйста, подожди...
✅ Модель успешно загружена!
Размерность векторов: 300
Количество слов в словаре: 184973
Примеры слов: ['весь_DET', 'человек_NOUN', 'мочь_VERB', 'год_NOUN', 'сказать_VERB', 'время_NOUN', 'говорить_VERB', 'становиться_VERB', 'знать_VERB', 'самый_DET']

Формат слов: слово_ЧАСТЬ_РЕЧИ (например, 'работа_NOUN')


In [10]:
"""
model = gensim.models.KeyedVectors.load_word2vec_format('web_0_300_20.bin', encoding='utf-8', unicode_errors='ignore', binary=True)
model.init_sims(replace=True)
"""

"\nmodel = gensim.models.KeyedVectors.load_word2vec_format('web_0_300_20.bin', encoding='utf-8', unicode_errors='ignore', binary=True)\nmodel.init_sims(replace=True)\n"

In [5]:
model.most_similar('работа_NOUN')

[('работать_VERB', 0.6488216519355774),
 ('пуско::наладочный_ADJ', 0.5520831942558289),
 ('наладочный_ADJ', 0.5179657936096191),
 ('общестроительный_ADJ', 0.5123682618141174),
 ('учеба_NOUN', 0.5118712782859802),
 ('деятельность_NOUN', 0.5090660452842712),
 ('грант::рффи_NOUN', 0.5028563737869263),
 ('работник_NOUN', 0.5006853938102722),
 ('тогр_NOUN', 0.4930512309074402),
 ('односменный_ADJ', 0.49021145701408386)]

In [6]:
model.key_to_index['работа_NOUN']

29

In [7]:
model[model.key_to_index['работа_NOUN']]

array([ 0.07628661, -0.05659969,  0.01138032,  0.14302921,  0.01386698,
        0.05855463, -0.05062321,  0.0705623 ,  0.03634923,  0.01049339,
        0.0111809 , -0.09291459, -0.06297037,  0.0043148 ,  0.07975311,
        0.0220203 , -0.02680903, -0.0015872 , -0.01891638,  0.06782877,
        0.11464189,  0.08125766, -0.09224732, -0.00629267,  0.03127897,
       -0.01073202,  0.09775175, -0.04135708, -0.01009073, -0.0816955 ,
       -0.08814938,  0.12237391,  0.01558208, -0.03481785,  0.01036572,
        0.00641243,  0.03946844,  0.12571745,  0.14080346, -0.11770941,
       -0.05367472, -0.03863762, -0.02049154, -0.12687334, -0.0585563 ,
        0.01127036,  0.07406449, -0.01001787,  0.10331652,  0.01166762,
       -0.02474485,  0.0418124 , -0.02196709,  0.01646761,  0.06897218,
       -0.02478149, -0.04773564, -0.02332791,  0.04762999,  0.00517535,
       -0.0753234 , -0.0256147 ,  0.0637579 ,  0.09777023, -0.07380333,
        0.01135527,  0.0009746 , -0.05738696,  0.02251336, -0.01

In [14]:
model.similarity('кошка_NOUN', 'собака_NOUN')

0.69631803

In [15]:
model.similarity('кошка_NOUN', 'день_NOUN')

0.1609627

In [16]:
# Аналогия: Франция = Париж + (Германия - Берлин)
model.most_similar(positive=['париж_NOUN','германия_NOUN'], negative=['берлин_NOUN'], topn=1)

[('франция_NOUN', 0.8673800230026245)]

In [22]:
# Сначала загружаем созданный файл
import pickle

with open('sentiment_POS', 'rb') as f:
    loaded_data = pickle.load(f)

print("Файл загружен успешно!")
print(f"Размер данных: {loaded_data.shape}")
print(f"Колонки: {loaded_data.columns.tolist()}")

# Теперь можно использовать loaded_data
X_tokens = loaded_data['X'].tolist()
y_labels = loaded_data['ttype'].values

print(f"\nГотово! Получено {len(X_tokens)} твитов с частеречной разметкой")
print(f"Пример первого твита (первые 10 токенов):")
print(X_tokens[0][:10])

# Проверим распределение меток
print(f"\nРаспределение тональностей:")
print(f"Негативных (0): {sum(y_labels == 0)}")
print(f"Позитивных (1): {sum(y_labels == 1)}")

Файл загружен успешно!
Размер данных: (226834, 4)
Колонки: ['ttext', 'ttype', 'text_pos', 'X']

Готово! Получено 226834 твитов с частеречной разметкой
Пример первого твита (первые 10 токенов):
['работе_NOUN', 'полный_ADJ', 'пиддес_NOUN', 'каждое_ADJ', 'закрытие_NOUN', 'месяца_NOUN', 'свихнусь_VERB']

Распределение тональностей:
Негативных (0): 111923
Позитивных (1): 114911


In [29]:
# Используем созданный файл для Word2Vec
X_tokens = loaded_data['X'].tolist()
y = loaded_data['ttype'].values  # переименуй в y, а не y_labels

print(f"Готово! Получено {len(X_tokens)} твитов с частеречной разметкой")
print(f"Пример первого твита (первые 10 токенов):")
print(X_tokens[0][:10])

def word_averaging(model, words):
    all_words, mean = set(), []

    for word in words:
        if word in model.key_to_index.keys():
            mean.append(model[model.key_to_index[word]])
            all_words.add(model.key_to_index[word])

    if not mean:
        #logging.warning("cannot compute similarity with no input %s", words)
        # FIXME: remove these examples in pre-processing
        return np.zeros(model.vector_size)

    mean = gensim.matutils.unitvec(np.array(mean).mean(axis=0)).astype(np.float32)
    return mean

def word_averaging_list(model, text_list):
    return np.vstack([word_averaging(model, comment_text) for comment_text in text_list])

# Получаем векторы для всех твитов
X = word_averaging_list(model, X_tokens)  # используем X_tokens, а не loaded_data['X']
print(f"Размер матрицы признаков X: {X.shape}")
print(f"Размер меток y: {y.shape}")

# Теперь разделяем на train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=0,
    stratify=y  # сохраняем пропорцию классов
)

print(f"\nРезультат разделения:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, 0: {sum(y_train==0)}, 1: {sum(y_train==1)}")
print(f"y_test: {y_test.shape}, 0: {sum(y_test==0)}, 1: {sum(y_test==1)}")

Готово! Получено 226834 твитов с частеречной разметкой
Пример первого твита (первые 10 токенов):
['работе_NOUN', 'полный_ADJ', 'пиддес_NOUN', 'каждое_ADJ', 'закрытие_NOUN', 'месяца_NOUN', 'свихнусь_VERB']
Размер матрицы признаков X: (226834, 300)
Размер меток y: (226834,)

Результат разделения:
X_train: (158783, 300)
X_test: (68051, 300)
y_train: (158783,), 0: 78346, 1: 80437
y_test: (68051,), 0: 33577, 1: 34474


In [30]:
def word_averaging(model, words):
    all_words, mean = set(), []

    for word in words:
        if word in model.key_to_index.keys():
            mean.append(model[model.key_to_index[word]])
            all_words.add(model.key_to_index[word])

    if not mean:
        #logging.warning("cannot compute similarity with no input %s", words)
        # FIXME: remove these examples in pre-processing
        return np.zeros(model.vector_size)

    mean = gensim.matutils.unitvec(np.array(mean).mean(axis=0)).astype(np.float32)
    return mean

In [31]:
def  word_averaging_list(model, text_list):
    return np.vstack([word_averaging(model, comment_text) for comment_text in text_list ])


In [32]:
X=loaded_data['X']

In [33]:
X = word_averaging_list(model, X)
X.shape

(226834, 300)

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [35]:
inputs_train = torch.tensor(X_train)
targets_train = torch.IntTensor(y_train)

inputs_test = torch.tensor(X_test)
targets_test = torch.IntTensor(y_test)

train = data_utils.TensorDataset(inputs_train, targets_train)
test = data_utils.TensorDataset(inputs_test, targets_test)

trainset = torch.utils.data.DataLoader(train, shuffle=True)
testset = torch.utils.data.DataLoader(test, shuffle=False)

In [36]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()  #вх. #вых.
        self.fc1 = nn.Linear(300, 128)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, 2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

net = Net()
print(net)

Net(
  (fc1): Linear(in_features=300, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=2, bias=True)
)


In [37]:
import torch.optim as optim

loss_function = nn.NLLLoss()# вычисляет, насколько далеко результаты нашей классификации отошли от реальности
optimizer = optim.Adam(net.parameters(), lr=0.001) # регулирует изменяемые параметры нашей модели, а именно веса, чтобы они постепенно, с течением времени, приходили в соответствие нашим данным
# lr - скорость обучения определяет величину изменений, которую оптимайзер может внести за один раз
# Каждый полный проход по данным называется эпохой (epoch)

In [40]:
for epoch in range(10): # три полных прохода по нашим данным
    for data in trainset:  # `data` это батч наших данных
        X, y = data  # X это батч свойств, y это батч целевых переменных.
        net.zero_grad()  # устанавливаем значение градиента в 0 перед вычислением функции потерь. Вам следует делать это на каждом шаге.
        output = net(X.float())  # передаем батч
        loss = loss_function(output, y.long())  # вычисляем функцию потерь
        loss.backward()  # передаем это значение назад по сети
        optimizer.step()  # пытаемся оптимизировать значение весов исходя из потерь и градиента
    print(loss)  # выводим на экран значение функции потерь. Мы надеемся, что оно убывает!

tensor(0.6922, grad_fn=<NllLossBackward0>)
tensor(0.0153, grad_fn=<NllLossBackward0>)
tensor(0.3716, grad_fn=<NllLossBackward0>)
tensor(0.2030, grad_fn=<NllLossBackward0>)
tensor(0.7109, grad_fn=<NllLossBackward0>)
tensor(0.7091, grad_fn=<NllLossBackward0>)
tensor(0.6662, grad_fn=<NllLossBackward0>)
tensor(0.1111, grad_fn=<NllLossBackward0>)
tensor(0.7047, grad_fn=<NllLossBackward0>)
tensor(0.5083, grad_fn=<NllLossBackward0>)


In [41]:
ams = []
with torch.no_grad():
    for data in testset:
        X, y = data
        output = net(X.float())
        for idx, i in enumerate(output):
            ams.append(torch.argmax(i).item())

In [42]:
print(output.size())
print(len(ams))

print(classification_report(y_test, ams))

torch.Size([1, 2])
68051
              precision    recall  f1-score   support

           0       0.66      0.43      0.52     33545
           1       0.59      0.78      0.67     34506

    accuracy                           0.61     68051
   macro avg       0.62      0.61      0.60     68051
weighted avg       0.62      0.61      0.60     68051

